# Relax 优化示例集

本 notebook 包含 6 个独立的优化示例，每个示例都可以单独运行。

## 目录

1. [基础变换流水线](#example1) - 演示最常用的优化 Pass 序列
2. [自定义 Pass - 算子替换](#example2) - 演示如何编写自定义变换 Pass
3. [内存优化](#example3) - 演示内存相关的优化 Pass
4. [布局优化 (CPU)](##example4) - 演示针对 CPU 的布局转换优化
5. [混合精度优化](#example5) - 演示 FP32 → FP16 的精度转换
6. [高级融合模式](#example6) - 演示高级融合技术

---

## 环境设置

运行以下单元格来设置环境。

In [1]:
# 环境设置
import sys
import os
import time
import numpy as np
from typing import Dict, List, Any, Tuple

# 设置环境变量
os.environ['TVM_HOME'] = '/home/lhy/tvm'
sys.path.insert(0, '/home/lhy/tvm/python')
sys.path.insert(0, '/home/lhy/tvm/ffi/python')

# 导入 TVM 和相关模块
import tvm
from tvm import IRModule, relax
from tvm.relax.frontend import nn
import tvm.relax.transform as transform

# 验证导入
print(f"TVM version: {tvm.__version__}")
print("Environment setup complete!")
print(f"NumPy: {np.__version__}")

TVM version: 0.23.dev0
Environment setup complete!
NumPy: 2.4.2


### 工具函数定义

In [ ]:
def apply_passes(mod: IRModule, passes: List, verbose: bool = False) -> IRModule:
    """应用一系列 Pass 到模块"""
    result = mod
    for i, pass_func in enumerate(passes):
        if verbose:
            pass_name = getattr(pass_func, '__name__', str(pass_func))
            print(f"  Applying pass {i+1}/{len(passes)}: {pass_name}")
        try:
            result = pass_func(result)
        except Exception as e:
            if verbose:
                print(f"    Warning: Pass failed with {type(e).__name__}: {e}")
    return result


def count_fused_kernels(mod: IRModule) -> Dict[str, Any]:
    """统计模块中融合 kernel 的数量和类型"""
    stats = {
        "total_functions": 0,
        "fused_functions": 0,
        "prim_functions": 0,
        "relax_functions": 0,
        "fused_names": []
    }
    
    for gvar, func in mod.functions_items():
        stats["total_functions"] += 1
        
        if hasattr(func, "attrs") and func.attrs is not None:
            if "prim_func_name" in func.attrs:
                stats["prim_functions"] += 1
        
        func_name = gvar.name_hint
        if "fused_" in func_name:
            stats["fused_functions"] += 1
            stats["fused_names"].append(func_name)
        
        if isinstance(func, relax.Function):
            stats["relax_functions"] += 1
    
    return stats


def print_module_stats(mod: IRModule, name: str = "Module"):
    """打印模块的统计信息"""
    stats = count_fused_kernels(mod)
    print(f"\n{name} Statistics:")
    print(f"  Total functions: {stats['total_functions']}")
    print(f"  Relax functions: {stats['relax_functions']}")
    print(f"  TIR PrimFuncs: {stats['prim_functions']}")
    print(f"  Fused functions: {stats['fused_functions']}")
    if stats['fused_names']:
        print(f"  Fused kernel names: {', '.join(stats['fused_names'][:5])}")
        if len(stats['fused_names']) > 5:
            print(f"    ... and {len(stats['fused_names']) - 5} more")
    return stats


def benchmark_module(
    mod: IRModule,
    params: Dict[str, np.ndarray],
    data_shape: Tuple[int, ...],
    target: str = "llvm",
    num_iterations: int = 10,
    warmup_iterations: int = 10
) -> Dict[str, Any]:
    """对模块进行性能测试"""
    device = tvm.cpu()
    
    # 生成随机测试数据
    data_np = np.random.randn(*data_shape).astype(np.float32)
    
    # 构建模块
    build_start = time.time()
    try:
        ex = tvm.compile(mod, target=target)
        build_time = time.time() - build_start
    except Exception as e:
        return {"error": str(e), "build_failed": True}
    
    # 创建虚拟机
    vm = relax.VirtualMachine(ex, device)
    
    # 转换数据为 TVM NDArray
    data_nd = tvm.nd.array(data_np, device=device)
    params_nd = {k: tvm.nd.array(v, device=device) for k, v in params.items()}
    
    # 预热
    for _ in range(warmup_iterations):
        _ = vm["forward"](data_nd, **params_nd)
    device.sync()
    
    # 性能测试
    times = []
    for _ in range(num_iterations):
        start = time.time()
        result = vm["forward"](data_nd, **params_nd)
        device.sync()
        end = time.time()
        times.append(end - start)
    
    times = np.array(times)
    
    return {
        "avg_time_ms": np.mean(times) * 1000,
        "std_time_ms": np.std(times) * 1000,
        "min_time_ms": np.min(times) * 1000,
        "max_time_ms": np.max(times) * 1000,
        "throughput": num_iterations / np.sum(times),
        "build_time": build_time,
        "build_failed": False
    }


print("Tool functions defined successfully!")

Tool functions defined successfully!


---

<a id='example1'></a>

## 示例 1: 基础变换流水线

**目标**: 演示最常用的优化 Pass 序列及其效果

### 步骤:
1. 创建一个简单的 MLP 模型
2. 应用标准优化流水线
3. 对比优化前后的性能

### 1.1 创建 MLP 模型

In [3]:
class MLP(nn.Module):
    """简单的多层感知机"""
    def __init__(self, input_dim=784, hidden_dim=256, output_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

# 创建并导出模型
model = MLP()
origin_mod, params = model.export_tvm(
    {"forward": {"x": nn.spec.Tensor(("n", 784), "float32")}}
)

print("Original MLP Model:")
origin_mod.show()
print_module_stats(origin_mod, "Original")

Original MLP Model:



Original Statistics:
  Total functions: 1
  Relax functions: 1
  TIR PrimFuncs: 0
  Fused functions: 0


{'total_functions': 1,
 'fused_functions': 0,
 'prim_functions': 0,
 'relax_functions': 1,
 'fused_names': []}

### 1.2 应用标准优化流水线

In [4]:
# 定义标准优化流水线
standard_pipeline = tvm.ir.transform.Sequential([
    # 1. 降低高级算子
    transform.LegalizeOps(),
    # 2. 简化和规范化
    transform.CanonicalizeBindings(),
    transform.FoldConstant(),
    # 3. 融合优化
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
    # 4. 死代码消除
    transform.DeadCodeElimination(),
])

# 应用优化流水线
optimized_mod = standard_pipeline(origin_mod)

print("\nOptimized Module:")
optimized_mod.show()
print_module_stats(optimized_mod, "Optimized")


Optimized Module:



Optimized Statistics:
  Total functions: 7
  Relax functions: 1
  TIR PrimFuncs: 0
  Fused functions: 3
  Fused kernel names: fused_matmul1_add_relu, fused_matmul2_add1, fused_matmul_add_relu


{'total_functions': 7,
 'fused_functions': 3,
 'prim_functions': 0,
 'relax_functions': 1,
 'fused_names': ['fused_matmul1_add_relu',
  'fused_matmul2_add1',
  'fused_matmul_add_relu']}

In [5]:
#单次执行对比


### 1.3 性能对比

In [6]:
# 测试原始模块性能
print("Benchmarking original module...")
original_stats = benchmark_module(origin_mod, params, (32, 784))

# 测试优化模块性能
print("Benchmarking optimized module...")
optimized_stats = benchmark_module(optimized_mod, params, (32, 784))

# 打印对比结果
print("\n" + "="*60)
print("PERFORMANCE COMPARISON")
print("="*60)
print(f"{'Metric':<25} {'Original':>15} {'Optimized':>15} {'Speedup':>10}")
print("-"*60)

if not original_stats.get("build_failed"):
    print(f"{'Avg Time (ms)':<25} {original_stats['avg_time_ms']:>15.4f} ", end="")
else:
    print(f"{'Avg Time (ms)':<25} {'N/A':>15} ", end="")

if not optimized_stats.get("build_failed"):
    print(f"{optimized_stats['avg_time_ms']:>15.4f} ", end="")
    if not original_stats.get("build_failed"):
        speedup = original_stats['avg_time_ms'] / optimized_stats['avg_time_ms']
        print(f"{speedup:>10.2f}x")
    else:
        print(f"{'N/A':>10}")
else:
    print(f"{'N/A':>15} {'N/A':>10}")

if not original_stats.get("build_failed") and not optimized_stats.get("build_failed"):
    print(f"\nThroughput: {original_stats['throughput']:.2f} → {optimized_stats['throughput']:.2f} inferences/sec")
    print(f"Build time: {original_stats['build_time']:.4f}s → {optimized_stats['build_time']:.4f}s")

Benchmarking original module...


: 

### 1.4 逐步查看每个 Pass 的效果

In [ ]:
# 逐步应用 Pass 并查看效果
passes = [
    ("LegalizeOps", transform.LegalizeOps()),
    ("CanonicalizeBindings", transform.CanonicalizeBindings()),
    ("FoldConstant", transform.FoldConstant()),
    ("AnnotateTIROpPattern", transform.AnnotateTIROpPattern()),
    ("FuseOps", transform.FuseOps()),
    ("FuseTIR", transform.FuseTIR()),
    ("DeadCodeElimination", transform.DeadCodeElimination()),
]

current_mod = origin_mod
print("\nStep-by-step Pass Application:\n")

for i, (name, pass_func) in enumerate(passes):
    print(f"\n{i+1}. Applying {name}...")
    current_mod = pass_func(current_mod)
    stats = print_module_stats(current_mod, f"After {name}")

---

<a id='example2'></a>

## 示例 2: 自定义 Pass - 算子替换

**目标**: 演示如何编写自定义变换 Pass

### 步骤:
1. 定义一个 `SigmoidToSwish` Mutator
2. 创建对应的 Pass 包装器
3. 应用到模型并验证效果

### 2.1 创建包含 Sigmoid 的模型

In [ ]:
import torch
class ModelWithSigmoid(nn.Module):
    """包含 Sigmoid 激活的模型"""
    def __init__(self, input_dim=784, hidden_dim=128, output_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.sigmoid1 = nn.Sigmoid()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.sigmoid1(x)
        x = self.fc2(x)
        return x

# 创建模型
sigmoid_model = ModelWithSigmoid()
sigmoid_mod, sigmoid_params = sigmoid_model.export_tvm(
    {"forward": {"x": nn.spec.Tensor(("n", 784), "float32")}}
)

print("Model with Sigmoid:")
sigmoid_mod.show()

### 2.2 定义自定义 Mutator - SigmoidToSwish

In [ ]:
from tvm.relax.expr_functor import PyExprMutator, mutator

@mutator
class SigmoidToSwishMutator(PyExprMutator):
    """
    将 Sigmoid 替换为 Swish 的 Mutator
    
    Swish(x) = x * Sigmoid(x)
    这是 Sigmoid 的一个变体，通常性能更好
    """
    def __init__(self, mod: IRModule):
        super().__init__(mod)
    
    def visit_call_(self, call: relax.Call) -> relax.Expr:
        """当遍历到 Call 节点时调用"""
        # 首先递归处理子节点
        call = super().visit_call_(call)
        
        # 检查是否是 sigmoid 算子
        if call.op.name == "relax.nn.sigmoid":
            x = call.args[0]
            # 替换为 swish: x * sigmoid(x)
            # 注意：这里我们实际上创建了一个使用 sigmoid 的 swish
            # 在实际应用中，可能需要更复杂的替换逻辑
            sigmoid_result = relax.op.nn.sigmoid(x)
            swish_result = relax.op.multiply(x, sigmoid_result)
            return swish_result
        
        return call

print("SigmoidToSwishMutator defined successfully!")

### 2.3 定义 Pass 包装器

In [ ]:
@tvm.transform.module_pass(opt_level=1, name="SigmoidToSwish")
class SigmoidToSwish:
    """
    将 Sigmoid 替换为 Swish 的 Pass
    
    这个 Pass 遍历模块中的所有函数，
    将所有的 sigmoid 调用替换为 swish 调用
    """
    def transform_module(
        self, 
        mod: IRModule, 
        _ctx: tvm.transform.PassContext
    ) -> IRModule:
        """模块级变换入口"""
        # 创建 mutator
        mutator = SigmoidToSwishMutator(mod)
        
        # 遍历模块中的所有函数
        for g_var, func in mod.functions_items():
            # 只处理 Relax 函数
            if isinstance(func, relax.Function):
                # 应用变换
                new_func = mutator.visit_expr(func)
                # 更新模块中的函数
                mutator.builder_.update_func(g_var, new_func)
        
        # 返回变换后的模块
        return mutator.builder_.get()

print("SigmoidToSwish Pass defined successfully!")

### 2.4 应用自定义 Pass

In [ ]:
# 应用自定义 Pass
swish_mod = SigmoidToSwish()(sigmoid_mod)

print("After applying SigmoidToSwish Pass:")
swish_mod.show()

### 2.5 将自定义 Pass 加入标准流水线

In [ ]:
# 创建包含自定义 Pass 的完整流水线
custom_pipeline = tvm.ir.transform.Sequential([
    SigmoidToSwish(),           # 自定义 Pass
    transform.LegalizeOps(),    # 降低算子
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
    transform.DeadCodeElimination(),
])

# 应用流水线
final_mod = custom_pipeline(sigmoid_mod)

print("Final module after custom pipeline:")
final_mod.show()
print_module_stats(final_mod, "Final")

### 2.6 另一个自定义 Pass 示例 - ReLU 到 GELU

In [ ]:
# 定义 ReLU 到 GELU 的 Mutator
@mutator
class ReluToGeluMutator(PyExprMutator):
    """将 ReLU 替换为 GELU 的 Mutator"""
    
    def __init__(self, mod: IRModule):
        super().__init__(mod)
    
    def visit_call_(self, call: relax.Call) -> relax.Expr:
        call = super().visit_call_(call)
        
        # 检查是否是 relu 算子
        if call.op.name == "relax.nn.relu":
            # 替换为 gelu
            return relax.op.nn.gelu(call.args[0])
        
        return call

# 定义 Pass
@tvm.transform.module_pass(opt_level=1, name="ReluToGelu")
class ReluToGelu:
    def transform_module(
        self, 
        mod: IRModule, 
        _ctx: tvm.transform.PassContext
    ) -> IRModule:
        mutator = ReluToGeluMutator(mod)
        for g_var, func in mod.functions_items():
            if isinstance(func, relax.Function):
                new_func = mutator.visit_expr(func)
                mutator.builder_.update_func(g_var, new_func)
        return mutator.builder_.get()

# 测试
test_mod = origin_mod  # 使用之前创建的 MLP 模型
gelu_mod = ReluToGelu()(test_mod)

print("After ReluToGelu transformation:")
gelu_mod.show()

---

<a id='example3'></a>

## 示例 3: 内存优化

**目标**: 演示内存相关的优化 Pass

### 步骤:
1. 创建一个内存密集型模型
2. 应用内存优化 Pass
3. 对比内存使用情况

### 3.1 创建内存密集型模型 (多层 CNN)

In [ ]:
class DeepCNN(nn.Module):
    """深度 CNN 模型 - 内存密集型"""
    def __init__(self):
        super().__init__()
        # 多个卷积层会产生大量中间结果
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.relu4 = nn.ReLU()
        self.fc = nn.Linear(256 * 32 * 32, 10)  # 假设输入是 32x32
    
    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.relu3(self.conv3(x))
        x = self.relu4(self.conv4(x))
        x = relax.op.reshape(x, (x.shape[0], -1))
        x = self.fc(x)
        return x

# 创建模型
cnn_model = DeepCNN()
cnn_mod, cnn_params = cnn_model.export_tvm(
    {"forward": {"x": nn.spec.Tensor(("n", 3, 32, 32), "float32")}}
)

print("Deep CNN Model (Memory Intensive):")
print_module_stats(cnn_mod, "DeepCNN")

### 3.2 应用内存优化 Pass

In [ ]:
# 内存优化流水线
memory_pipeline = tvm.ir.transform.Sequential([
    # 首先降低算子
    transform.LegalizeOps(),
    
    # 融合优化 (减少中间结果)
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
    
    # 内存优化 Pass
    transform.StaticPlanBlockMemory(),  # 静态内存规划
    transform.KillAfterLastUse(),       # 及时释放内存
    transform.LiftTransformParams(),    # 参数提升
    
    # 死代码消除
    transform.DeadCodeElimination(),
])

# 应用内存优化
memory_optimized_mod = memory_pipeline(cnn_mod)

print("\nAfter Memory Optimization:")
print_module_stats(memory_optimized_mod, "Memory Optimized")

### 3.3 逐步查看内存优化 Pass 的效果

In [ ]:
# 先降低算子
legalized_mod = transform.LegalizeOps()(cnn_mod)

# 查看融合前后的差异
print("\n=== Before Fusion ===")
print_module_stats(legalized_mod, "Legalized")

# 应用融合
fused_mod = tvm.ir.transform.Sequential([
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
])(legalized_mod)

print("\n=== After Fusion ===")
print_module_stats(fused_mod, "Fused")

# 应用内存规划
memory_planned_mod = transform.StaticPlanBlockMemory()(fused_mod)

print("\n=== After Memory Planning ===")
print_module_stats(memory_planned_mod, "Memory Planned")

### 3.4 对比有无内存优化的性能

In [ ]:
# 创建无内存优化的版本
no_memory_opt = tvm.ir.transform.Sequential([
    transform.LegalizeOps(),
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
])(cnn_mod)

# 性能测试
print("Benchmarking without memory optimization...")
no_mem_stats = benchmark_module(no_memory_opt, cnn_params, (4, 3, 32, 32))

print("Benchmarking with memory optimization...")
with_mem_stats = benchmark_module(memory_optimized_mod, cnn_params, (4, 3, 32, 32))

# 打印结果
print("\n" + "="*60)
print("MEMORY OPTIMIZATION COMPARISON")
print("="*60)

if not no_mem_stats.get("build_failed") and not with_mem_stats.get("build_failed"):
    print(f"\nWithout Memory Opt:")
    print(f"  Avg Time: {no_mem_stats['avg_time_ms']:.4f} ms")
    print(f"  Throughput: {no_mem_stats['throughput']:.2f} inf/sec")
    
    print(f"\nWith Memory Opt:")
    print(f"  Avg Time: {with_mem_stats['avg_time_ms']:.4f} ms")
    print(f"  Throughput: {with_mem_stats['throughput']:.2f} inf/sec")
    
    speedup = no_mem_stats['avg_time_ms'] / with_mem_stats['avg_time_ms']
    print(f"\nSpeedup: {speedup:.2f}x")
else:
    print("Benchmark failed - check error messages above")

---

<a id='example4'></a>

## 示例 4: 布局优化 (CPU)

**目标**: 演示针对 CPU 的布局转换优化 (NCHW → NHWC)

### 步骤:
1. 创建包含 Conv2D 的模型 (默认 NCHW)
2. 应用 ConvertLayout 转换为 NHWC
3. 对比 CPU 上的性能差异

### 4.1 创建 Conv2D 模型

In [ ]:
class SimpleCNN(nn.Module):
    """简单的 CNN 用于演示布局转换"""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool = nn.MaxPool2d(pool_size=2)
        self.fc = nn.Linear(32 * 16 * 16, 10)
    
    def forward(self, x):
        # 默认 NCHW 布局
        x = self.relu1(self.conv1(x))
        x = self.pool(x)
        x = self.relu2(self.conv2(x))
        x = self.pool(x)
        x = relax.op.reshape(x, (x.shape[0], -1))
        x = self.fc(x)
        return x

# 创建模型
conv_model = SimpleCNN()
conv_mod, conv_params = conv_model.export_tvm(
    {"forward": {"x": nn.spec.Tensor(("n", 3, 32, 32), "float32")}}
)

print("Simple CNN (NCHW layout):")
print_module_stats(conv_mod, "CNN")

### 4.2 应用布局转换 (NCHW → NHWC)

In [ ]:
# 定义目标布局
# NHWC 对 CPU 更友好（内存连续）
desired_layouts = {
    "relax.nn.conv2d": ["NHWC", "OHWI"],
}

# 布局优化流水线
layout_pipeline_nchw = tvm.ir.transform.Sequential([
    transform.LegalizeOps(),
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
    transform.DeadCodeElimination(),
])

layout_pipeline_nhwc = tvm.ir.transform.Sequential([
    # 布局转换
    transform.ConvertLayout(desired_layouts),
    
    # 降低和优化
    transform.LegalizeOps(),
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
    transform.DeadCodeElimination(),
])

# 应用两种流水线
nchw_mod = layout_pipeline_nchw(conv_mod)
nhwc_mod = layout_pipeline_nhwc(conv_mod)

print("\n=== NCHW Layout (Default for GPU) ===")
print_module_stats(nchw_mod, "NCHW")

print("\n=== NHWC Layout (Optimized for CPU) ===")
print_module_stats(nhwc_mod, "NHWC")

### 4.3 查看布局转换后的 IR

In [ ]:
# 只应用 ConvertLayout 查看效果
layout_converted = transform.ConvertLayout(desired_layouts)(conv_mod)

print("After ConvertLayout (before legalization):")
layout_converted.show()

### 4.4 性能对比 (CPU)

In [ ]:
# 性能测试
print("Benchmarking NCHW layout...")
nchw_stats = benchmark_module(nchw_mod, conv_params, (8, 3, 32, 32))

print("Benchmarking NHWC layout...")
nhwc_stats = benchmark_module(nhwc_mod, conv_params, (8, 3, 32, 32))

# 打印结果
print("\n" + "="*60)
print("LAYOUT OPTIMIZATION COMPARISON (CPU)")
print("="*60)

if not nchw_stats.get("build_failed") and not nhwc_stats.get("build_failed"):
    print(f"\nNCHW (GPU-friendly):")
    print(f"  Avg Time: {nchw_stats['avg_time_ms']:.4f} ms")
    print(f"  Throughput: {nchw_stats['throughput']:.2f} inf/sec")
    
    print(f"\nNHWC (CPU-friendly):")
    print(f"  Avg Time: {nhwc_stats['avg_time_ms']:.4f} ms")
    print(f"  Throughput: {nhwc_stats['throughput']:.2f} inf/sec")
    
    speedup = nchw_stats['avg_time_ms'] / nhwc_stats['avg_time_ms']
    print(f"\nSpeedup: {speedup:.2f}x")
    
    if speedup > 1:
        print(f"\n✓ NHWC is {speedup:.2f}x faster on CPU!")
    else:
        print(f"\n✗ NCHW is {1/speedup:.2f}x faster on this run.")
else:
    print("Benchmark failed - check error messages above")

---

<a id='example5'></a>

## 示例 5: 混合精度优化

**目标**: 演示 FP32 → FP16 的精度转换

### 步骤:
1. 创建模型
2. 应用 ToMixedPrecision Pass
3. 验证精度损失和性能提升

### 5.1 创建测试模型

In [ ]:
class MixedPrecisionModel(nn.Module):
    """用于测试混合精度的模型"""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

# 创建模型
mp_model = MixedPrecisionModel()
mp_mod, mp_params = mp_model.export_tvm(
    {"forward": {"x": nn.spec.Tensor(("n", 784), "float32")}}
)

print("Model for Mixed Precision Testing:")
mp_mod.show()

### 5.2 应用混合精度转换

In [ ]:
# FP32 优化流水线
fp32_pipeline = tvm.ir.transform.Sequential([
    transform.LegalizeOps(),
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
])

# FP16 混合精度流水线
fp16_pipeline = tvm.ir.transform.Sequential([
    # 转换为混合精度
    transform.ToMixedPrecision(),
    
    # 后续优化
    transform.LegalizeOps(),
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
    transform.DeadCodeElimination(),
])

# 应用两种流水线
fp32_mod = fp32_pipeline(mp_mod)
fp16_mod = fp16_pipeline(mp_mod)

print("\n=== FP32 Module ===")
print_module_stats(fp32_mod, "FP32")

print("\n=== FP16 Mixed Precision Module ===")
print_module_stats(fp16_mod, "FP16")

### 5.3 查看混合精度转换后的 IR

In [ ]:
# 只应用 ToMixedPrecision 查看效果
mp_converted = transform.ToMixedPrecision()(mp_mod)

print("After ToMixedPrecision (before full optimization):")
mp_converted.show()

### 5.4 验证输出精度

In [ ]:
# 构建两个版本
device = tvm.cpu()

fp32_ex = relax.build(fp32_mod, target="llvm")
fp16_ex = relax.build(fp16_mod, target="llvm")

fp32_vm = relax.VirtualMachine(fp32_ex, device)
fp16_vm = relax.VirtualMachine(fp16_ex, device)

# 准备测试数据
test_data = np.random.randn(4, 784).astype(np.float32)
test_data_nd = tvm.nd.array(test_data, device=device)
params_nd = {k: tvm.nd.array(v, device=device) for k, v in mp_params.items()}

# 运行推理
fp32_result = fp32_vm["main"](test_data_nd, **params_nd).numpy()
fp16_result = fp16_vm["main"](test_data_nd, **params_nd).numpy()

# 计算精度差异
abs_diff = np.abs(fp32_result - fp16_result)
max_diff = np.max(abs_diff)
mean_diff = np.mean(abs_diff)
relative_error = mean_diff / (np.mean(np.abs(fp32_result)) + 1e-10)

print("\n" + "="*60)
print("PRECISION COMPARISON: FP32 vs FP16")
print("="*60)
print(f"\nFP32 output sample: {fp32_result[0][:5]}")
print(f"FP16 output sample: {fp16_result[0][:5]}")
print(f"\nAbsolute difference:")
print(f"  Max:  {max_diff:.6e}")
print(f"  Mean: {mean_diff:.6e}")
print(f"\nRelative error: {relative_error:.6e}")

if relative_error < 1e-3:
    print("\n✓ Precision loss is acceptable (< 0.1%)")
elif relative_error < 1e-2:
    print("\n⚠ Precision loss is moderate (< 1%)")
else:
    print("\n✗ Precision loss is significant!")

### 5.5 性能对比

In [ ]:
# 性能测试
print("Benchmarking FP32...")
fp32_perf = benchmark_module(fp32_mod, mp_params, (32, 784))

print("Benchmarking FP16...")
fp16_perf = benchmark_module(fp16_mod, mp_params, (32, 784))

# 打印结果
print("\n" + "="*60)
print("PERFORMANCE COMPARISON: FP32 vs FP16")
print("="*60)

if not fp32_perf.get("build_failed") and not fp16_perf.get("build_failed"):
    print(f"\nFP32:")
    print(f"  Avg Time: {fp32_perf['avg_time_ms']:.4f} ms")
    print(f"  Throughput: {fp32_perf['throughput']:.2f} inf/sec")
    
    print(f"\nFP16:")
    print(f"  Avg Time: {fp16_perf['avg_time_ms']:.4f} ms")
    print(f"  Throughput: {fp16_perf['throughput']:.2f} inf/sec")
    
    speedup = fp32_perf['avg_time_ms'] / fp16_perf['avg_time_ms']
    print(f"\nSpeedup: {speedup:.2f}x")
    
    # 内存使用估计
    fp32_memory = 784 * 256 + 256 * 128 + 128 * 10  # 权重大小
    fp16_memory = fp32_memory / 2
    print(f"\nMemory usage estimate:")
    print(f"  FP32: {fp32_memory / 1024:.2f} KB (weights only)")
    print(f"  FP16: {fp16_memory / 1024:.2f} KB (weights only)")
    print(f"  Memory saved: {(fp32_memory - fp16_memory) / fp32_memory * 100:.1f}%")
else:
    print("Benchmark failed - check error messages above")

---

<a id='example6'></a>

## 示例 6: 高级融合模式

**目标**: 演示高级融合技术

### 步骤:
1. 创建包含多个并行 MatMul 的模型
2. 应用 CombineParallelMatmul 合并并行操作
3. 应用 FuseTransposeMatmul 融合转置和矩阵乘法

### 6.1 创建包含并行 MatMul 的模型

In [ ]:
class ParallelMatMulModel(nn.Module):
    """
    包含多个并行矩阵乘法的模型
    这种模式在多头注意力等场景中很常见
    """
    def __init__(self, input_dim=256, output_dim=64, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        # 多个并行线性层
        self.projections = nn.ModuleList([
            nn.Linear(input_dim, output_dim) for _ in range(num_heads)
        ])
    
    def forward(self, x):
        # 并行矩阵乘法
        outputs = []
        for i in range(self.num_heads):
            outputs.append(self.projections[i](x))
        # 合并结果
        return relax.op.concat(outputs, axis=1)

# 创建模型
parallel_model = ParallelMatMulModel()
parallel_mod, parallel_params = parallel_model.export_tvm(
    {"forward": {"x": nn.spec.Tensor(("n", 256), "float32")}}
)

print("Model with Parallel MatMul:")
parallel_mod.show()
print_module_stats(parallel_mod, "Parallel")

### 6.2 应用 CombineParallelMatmul

In [ ]:
# 标准优化流水线（不包含并行合并）
standard_fusion = tvm.ir.transform.Sequential([
    transform.LegalizeOps(),
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
])

# 包含并行合并的流水线
parallel_fusion = tvm.ir.transform.Sequential([
    transform.LegalizeOps(),
    # 合并并行矩阵乘法
    transform.CombineParallelMatmul(),
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
])

# 应用两种流水线
standard_mod = standard_fusion(parallel_mod)
combined_mod = parallel_fusion(parallel_mod)

print("\n=== Standard Fusion ===")
print_module_stats(standard_mod, "Standard")

print("\n=== With CombineParallelMatmul ===")
print_module_stats(combined_mod, "Combined")

### 6.3 查看并行合并后的 IR

In [ ]:
# 只应用 CombineParallelMatmul 查看效果
legalized_parallel = transform.LegalizeOps()(parallel_mod)
combined_parallel = transform.CombineParallelMatmul()(legalized_parallel)

print("After CombineParallelMatmul (before fusion):")
combined_parallel.show()

### 6.4 创建包含 Transpose + MatMul 的模型

In [ ]:
class TransposeMatMulModel(nn.Module):
    """
    包含 Transpose + MatMul 模式的模型
    这种模式在 Attention 机制中很常见
    """
    def __init__(self, dim=64):
        super().__init__()
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.out = nn.Linear(dim, dim)
    
    def forward(self, x):
        # Q, K, V 投影
        q = self.query(x)  # (B, L, D)
        k = self.key(x)    # (B, L, D)
        v = self.value(x)  # (B, L, D)
        
        # 转置 K 用于矩阵乘法
        k_transposed = relax.op.transpose(k, axes=[0, 2, 1])
        
        # Attention scores: Q @ K^T
        attn = relax.op.matmul(q, k_transposed)
        
        # Softmax
        attn = relax.op.nn.softmax(attn, axis=-1)
        
        # Apply to V
        out = relax.op.matmul(attn, v)
        
        # Output projection
        return self.out(out)

# 创建模型
transpose_model = TransposeMatMulModel()
transpose_mod, transpose_params = transpose_model.export_tvm(
    {"forward": {"x": nn.spec.Tensor(("n", "l", 64), "float32")}}
)

print("Model with Transpose + MatMul:")
print_module_stats(transpose_mod, "TransposeMatMul")

### 6.5 应用 FuseTransposeMatmul

In [ ]:
# 标准流水线
standard_transpose = tvm.ir.transform.Sequential([
    transform.LegalizeOps(),
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
])

# 包含转置融合的流水线
transpose_fusion = tvm.ir.transform.Sequential([
    transform.LegalizeOps(),
    # 融合转置和矩阵乘法
    transform.FuseTransposeMatmul(),
    transform.AnnotateTIROpPattern(),
    transform.FuseOps(),
    transform.FuseTIR(),
])

# 应用两种流水线
standard_t_mod = standard_transpose(transpose_mod)
fused_t_mod = transpose_fusion(transpose_mod)

print("\n=== Standard Fusion ===")
print_module_stats(standard_t_mod, "Standard")

print("\n=== With FuseTransposeMatmul ===")
print_module_stats(fused_t_mod, "Fused")

### 6.6 性能对比

In [ ]:
# 性能测试 - 并行 MatMul
print("Benchmarking standard fusion (parallel matmul)...")
standard_p_stats = benchmark_module(standard_mod, parallel_params, (16, 256))

print("Benchmarking with CombineParallelMatmul...")
combined_p_stats = benchmark_module(combined_mod, parallel_params, (16, 256))

# 打印结果
print("\n" + "="*60)
print("ADVANCED FUSION COMPARISON")
print("="*60)

if not standard_p_stats.get("build_failed") and not combined_p_stats.get("build_failed"):
    print(f"\nParallel MatMul:")
    print(f"  Standard:  {standard_p_stats['avg_time_ms']:.4f} ms")
    print(f"  Combined:  {combined_p_stats['avg_time_ms']:.4f} ms")
    speedup = standard_p_stats['avg_time_ms'] / combined_p_stats['avg_time_ms']
    print(f"  Speedup:   {speedup:.2f}x")
else:
    print("Parallel MatMul benchmark failed")

# 性能测试 - Transpose MatMul
print("\nBenchmarking standard fusion (transpose matmul)...")
standard_t_stats = benchmark_module(standard_t_mod, transpose_params, (4, 32, 64))

print("Benchmarking with FuseTransposeMatmul...")
fused_t_stats = benchmark_module(fused_t_mod, transpose_params, (4, 32, 64))

if not standard_t_stats.get("build_failed") and not fused_t_stats.get("build_failed"):
    print(f"\nTranspose MatMul:")
    print(f"  Standard: {standard_t_stats['avg_time_ms']:.4f} ms")
    print(f"  Fused:    {fused_t_stats['avg_time_ms']:.4f} ms")
    speedup = standard_t_stats['avg_time_ms'] / fused_t_stats['avg_time_ms']
    print(f"  Speedup:  {speedup:.2f}x")
else:
    print("Transpose MatMul benchmark failed")

---

## 总结

本 notebook 包含了 6 个独立的 Relax 优化示例：

1. **基础变换流水线** - 演示了标准优化 Pass 序列的效果
2. **自定义 Pass** - 展示了如何编写自定义变换 Pass
3. **内存优化** - 演示了内存相关优化 Pass 的使用
4. **布局优化** - 展示了 NCHW → NHWC 布局转换
5. **混合精度** - 演示了 FP32 → FP16 转换和精度验证
6. **高级融合** - 展示了 CombineParallelMatmul 和 FuseTransposeMatmul

### 关键要点

- **Pass 顺序很重要**: LegalizeOps → FuseOps → Memory Optimization
- **融合是最重要的优化**: 可以显著减少 kernel 启动开销
- **内存规划**: StaticPlanBlockMemory 可以有效减少内存使用
- **布局优化**: CPU 适合 NHWC，GPU 适合 NCHW
- **混合精度**: FP16 可以减少内存使用和提升性能，但可能有精度损失
- **高级融合**: CombineParallelMatmul 和 FuseTransposeMatmul 可以进一步优化特定模式